# PREPARATION

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip -q install -U transformers accelerate datasets peft bitsandbytes qwen-vl-utils

In [3]:
!pip uninstall -y torchao
!pip install -U peft

In [15]:
import os
import json
import math
import cv2


root_dir = "/content/drive/MyDrive/CholecTrack20/Testing"
output_root = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips"
clip_minutes = 15
min_keep_minutes = 10
# =========================

os.makedirs(output_root, exist_ok=True)

def find_one_file_by_ext(folder, exts):
    files = []
    for f in os.listdir(folder):
        lower = f.lower()
        if any(lower.endswith(ext) for ext in exts):
            files.append(os.path.join(folder, f))
    return sorted(files)

def save_video_clip(video_path, out_path, start_frame, end_frame, fps, width, height):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    current = start_frame
    while current <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        writer.write(frame)
        current += 1

    writer.release()
    cap.release()

def build_clip_json(data, clip_id, source_video_path, start_frame, end_frame, fps, width, height):
    annotations = data["annotations"]
    video_info = data.get("video", {})
    info = data.get("info", {})
    categories = data.get("categories", {})

    ann_keys = sorted([int(k) for k in annotations.keys()])


    ann_in_clip_original = {
        str(k): annotations[str(k)]
        for k in ann_keys
        if start_frame <= k <= end_frame
    }


    ann_in_clip_local = {}
    for k_str, ann_list in ann_in_clip_original.items():
        k = int(k_str)
        local_k = k - start_frame
        ann_in_clip_local[str(local_k)] = ann_list

    clip_json = {
        "clip_id": clip_id,
        "video_name": os.path.basename(source_video_path).rsplit(".", 1)[0],
        "source_video_path": source_video_path,

        "start_frame_original": int(start_frame),
        "end_frame_original": int(end_frame),

        "start_time_sec": float(start_frame / fps),
        "end_time_sec": float(end_frame / fps),

        "fps": float(fps),
        "width": int(width),
        "height": int(height),

        "num_frames_in_clip": int(end_frame - start_frame + 1),
        "num_annotated_frames": len(ann_in_clip_original),


        "annotated_frame_ids_original": [int(k) for k in ann_in_clip_original.keys()],
        "annotations_in_clip_original": ann_in_clip_original,

        "annotated_frame_ids_local": sorted([int(k) for k in ann_in_clip_local.keys()]),
        "annotations_in_clip_local": ann_in_clip_local,

        "info": info,
        "categories": categories,
        "source_video_meta": video_info,
    }
    return clip_json

subfolders = sorted([
    os.path.join(root_dir, d)
    for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d))
])

print("Found subfolders:", len(subfolders))

all_clip_count = 0

for folder in subfolders:
    folder_name = os.path.basename(folder)

    mp4_files = find_one_file_by_ext(folder, [".mp4"])
    json_files = find_one_file_by_ext(folder, [".json"])

    if len(mp4_files) == 0 or len(json_files) == 0:
        print(f"[SKIP] {folder_name}: missing mp4 or json")
        continue

    video_path = mp4_files[0]
    json_path = json_files[0]

    print("\n" + "=" * 80)
    print("Processing folder:", folder_name)
    print("video_path:", video_path)
    print("json_path:", json_path)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[SKIP] Cannot open video: {video_path}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    num_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if fps <= 0 or num_frames_video <= 0:
        print(f"[SKIP] Invalid video info for: {video_path}")
        continue

    clip_len_frames = int(clip_minutes * 60 * fps)
    min_keep_frames = int(min_keep_minutes * 60 * fps)

    full_clips = num_frames_video // clip_len_frames
    remainder = num_frames_video % clip_len_frames


    out_dir = os.path.join(output_root, folder_name)
    os.makedirs(out_dir, exist_ok=True)

    clip_ranges = []


    for i in range(full_clips):
        start_frame = i * clip_len_frames
        end_frame = (i + 1) * clip_len_frames - 1
        clip_ranges.append((start_frame, end_frame))


    if remainder > min_keep_frames:
        start_frame = full_clips * clip_len_frames
        end_frame = num_frames_video - 1
        clip_ranges.append((start_frame, end_frame))

    print("fps:", fps)
    print("num_frames_video:", num_frames_video)
    print("clip_len_frames:", clip_len_frames)
    print("remainder_frames:", remainder)
    print("num_output_clips:", len(clip_ranges))

    for i, (start_frame, end_frame) in enumerate(clip_ranges, start=1):
        clip_id = f"{folder_name}_clip_{i:02d}"
        clip_mp4_path = os.path.join(out_dir, clip_id + ".mp4")
        clip_json_path = os.path.join(out_dir, clip_id + ".json")

        save_video_clip(
            video_path=video_path,
            out_path=clip_mp4_path,
            start_frame=start_frame,
            end_frame=end_frame,
            fps=fps,
            width=width,
            height=height,
        )

        clip_json = build_clip_json(
            data=data,
            clip_id=clip_id,
            source_video_path=video_path,
            start_frame=start_frame,
            end_frame=end_frame,
            fps=fps,
            width=width,
            height=height,
        )

        with open(clip_json_path, "w", encoding="utf-8") as f:
            json.dump(clip_json, f, indent=2, ensure_ascii=False)

        print(f"[SAVED] {clip_mp4_path}")
        print(f"[SAVED] {clip_json_path}")
        all_clip_count += 1

print("\n" + "=" * 80)
print("All done.")
print("Total clips created:", all_clip_count)
print("Output root:", output_root)

Found subfolders: 8

Processing folder: VID01
video_path: /content/drive/MyDrive/CholecTrack20/Testing/VID01/VID01.mp4
json_path: /content/drive/MyDrive/CholecTrack20/Testing/VID01/VID01.json
fps: 25.0
num_frames_video: 45175
clip_len_frames: 22500
remainder_frames: 175
num_output_clips: 2
[SAVED] /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/VID01_clip_01.mp4
[SAVED] /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/VID01_clip_01.json
[SAVED] /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/VID01_clip_02.mp4
[SAVED] /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/VID01_clip_02.json

Processing folder: VID06
video_path: /content/drive/MyDrive/CholecTrack20/Testing/VID06/VID06.mp4
json_path: /content/drive/MyDrive/CholecTrack20/Testing/VID06/VID06.json
fps: 25.0
num_frames_video: 68141
clip_len_frames: 22500
remainder_frames: 641
num_output_clips: 3
[SAVED] /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/VID06_cli

# ZEROSHOT

In [32]:
import os
import json

test_dir = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing"
phase_out = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/test_phase_task.jsonl"
anchor_out = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/test_anchor_task.jsonl"


PRIOR_RULE = (
    "Use the following prior information and label mappings. "
    "These mappings are provided only to explain the meaning of each integer ID. "
    "In the final JSON output, you must use integer IDs only. "
    "Do not output category names, text labels, or descriptions in the final JSON. "
)

PHASE_PRIOR = (
    "Phase ID mapping: "
    "0=Preparation; "
    "1=Calot triangle dissection; "
    "2=Clipping and cutting; "
    "3=Gallbladder dissection; "
    "4=Gallbladder retraction; "
    "5=Cleaning and coagulation; "
    "6=Gallbladder packaging. "
)

INSTRUMENT_PRIOR = (
    "Instrument ID mapping: "
    "0=grasper; "
    "1=bipolar; "
    "2=hook; "
    "3=scissors; "
    "4=clipper; "
    "5=irrigator; "
    "6=specimen-bag. "
)

OPERATOR_PRIOR = (
    "Operator ID mapping: "
    "0=null; "
    "1=main-surgeon-left-hand; "
    "2=assistant-surgeon-right-hand; "
    "3=main-surgeon-right-hand. "
)
JSON_ONLY_RULE = (
    "Do not output markdown. "
    "Do not output code fences. "
    "Do not output explanations. "
    "Output JSON only. "

)

def build_phase_segments(sample):

    annotated_frame_ids = sorted(sample["annotated_frame_ids_local"])
    annotations_in_clip = sample["annotations_in_clip"]
    num_frames_in_clip = sample["num_frames_in_clip"]

    if len(annotated_frame_ids) == 0:
        return {"phase_segments": []}

    frame_phase_pairs = []
    for fid in annotated_frame_ids:
        ann_list = annotations_in_clip[str(fid)]
        phase = ann_list[0]["phase"] if len(ann_list) > 0 else -1
        frame_phase_pairs.append((fid, phase))

    segments = []
    current_phase = frame_phase_pairs[0][1]
    seg_start = 0

    for i in range(1, len(frame_phase_pairs)):
        curr_fid, curr_phase = frame_phase_pairs[i]

        if curr_phase != current_phase:
            seg_end = curr_fid - 1
            segments.append({
                "start_frame": int(seg_start),
                "end_frame": int(seg_end),
                "phase": int(current_phase)
            })
            seg_start = curr_fid
            current_phase = curr_phase

    segments.append({
        "start_frame": int(seg_start),
        "end_frame": int(num_frames_in_clip - 1),
        "phase": int(current_phase)
    })

    return {"phase_segments": segments}

def round_bbox(bbox, ndigits=4):
    return [round(float(x), ndigits) for x in bbox]


def build_phase_prompt(num_frames_in_clip):
    return (
        "<video>\n"
        f"{PRIOR_RULE}"
        f"{PHASE_PRIOR}"
        f"The video has frames indexed from 0 to {num_frames_in_clip - 1}. "
        "Return only a valid JSON object with exactly one key: 'phase_segments'. "
        "The value of 'phase_segments' must be a list of segments that cover the whole video "
        "from frame 0 to the last frame. "
        "Each segment must contain exactly these keys: "
        "'start_frame', 'end_frame', 'phase'. "
        "The segments must be continuous, non-overlapping, and sorted by time. "
        "The 'phase' value must be an integer ID only, using the phase ID mapping above. "
        "Do not use phase names or words in the final JSON. "
        "Do not return a list directly. "
        f"{JSON_ONLY_RULE}"
        "Return a JSON object of the form: "
        "{\"phase_segments\": [{\"start_frame\": 0, \"end_frame\": 100, \"phase\": 1}]}"
    )


def build_anchor_frames(sample, seconds_stride=10):
    annotations_in_clip = sample["annotations_in_clip"]
    annotated_frame_ids = sorted(sample["annotated_frame_ids_local"])
    fps = float(sample["fps"])
    num_frames_in_clip = int(sample["num_frames_in_clip"])

    if len(annotated_frame_ids) == 0:
        return {"frames": []}

    # 每 10 秒取一个目标帧
    target_frame_ids = []
    sec = 0
    while True:
        target_fid = int(round(sec * seconds_stride * fps))
        if target_fid >= num_frames_in_clip:
            break
        target_frame_ids.append(target_fid)
        sec += 1

    frames = []
    for target_fid in target_frame_ids:
        nearest_fid = min(annotated_frame_ids, key=lambda x: abs(x - target_fid))
        ann_list = annotations_in_clip[str(nearest_fid)]
        phase = ann_list[0]["phase"] if len(ann_list) > 0 else -1

        objects = []
        for ann in ann_list:
            objects.append({
                "instrument": int(ann["instrument"]),
                "tool_bbox": round_bbox(ann["tool_bbox"]),
                "operator": int(ann["operator"]),
                "intraoperative_track": int(ann["intraoperative_track"]),
                "intracorporeal_track": int(ann["intracorporeal_track"]),
            })

        frames.append({
            "frame_index": int(target_fid),
            "phase": int(phase),
            "objects": objects
        })

    return {"frames": frames}


def build_anchor_prompt(frame_indices):
    return (
        "<video>\n"
        f"{PRIOR_RULE}"
        f"{PHASE_PRIOR}"
        f"{INSTRUMENT_PRIOR}"
        f"{OPERATOR_PRIOR}"
        "Return only one valid JSON object. "
        "The top-level output must be a JSON object, not a JSON array. "
        "The JSON object must contain exactly one key: 'frames'. "
        f"The value of 'frames' must be a list containing exactly one item for each of these frame indices sampled every 10 seconds: {frame_indices}. "
        "Do not omit any requested frame index. "
        "Do not invent any extra frame index. "
        "Each item in 'frames' must be a JSON object with exactly these keys: "
        "'frame_index', 'phase', 'objects'. "
        "The 'frame_index' value must be an integer and must equal one of the requested frame indices. "
        "The 'phase' value must be an integer ID only, using the phase ID mapping above. "
        "The 'objects' value must be a list. "
        "If no object is visible in a requested frame, return 'objects': []. "
        "Each item in 'objects' must be a JSON object with exactly these keys: "
        "'instrument', 'tool_bbox', 'operator', 'intraoperative_track', 'intracorporeal_track'. "
        "Every object must include all five keys. "
        "The 'instrument' value must be an integer ID only, using the instrument ID mapping above. "
        "The 'operator' value must be an integer ID only, using the operator ID mapping above. "
        "The 'intraoperative_track' value must be an integer ID only. "
        "The 'intracorporeal_track' value must be an integer ID only. "
        "The value of 'tool_bbox' must be a list of exactly 4 numbers in normalized [x, y, w, h] format. "
        "Do not use class names or text labels in the final JSON. "
        f"{JSON_ONLY_RULE}"
    )


files = sorted(os.listdir(test_dir))
mp4_files = [f for f in files if f.endswith(".mp4") and "_clip_" in f]

phase_rows = []
anchor_rows = []

for mp4_name in mp4_files:
    base = os.path.splitext(mp4_name)[0]
    json_name = base + ".json"

    mp4_path = os.path.join(test_dir, mp4_name)
    json_path = os.path.join(test_dir, json_name)

    if not os.path.exists(json_path):
        print("[WARNING] missing json for:", mp4_name)
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        sample = json.load(f)

    if sample["num_annotated_frames"] == 0:
        print("[SKIP] no annotations:", json_name)
        continue

    # Prompt A
    phase_gt = build_phase_segments(sample)
    phase_rows.append({
        "video": mp4_path,
        "prompt": build_phase_prompt(sample["num_frames_in_clip"]),
        "ground_truth": phase_gt,
        "clip_json_path": json_path
    })

    # Prompt B
    anchor_gt = build_anchor_frames(sample, seconds_stride=10)
    frame_indices = [x["frame_index"] for x in anchor_gt["frames"]]

    anchor_rows.append({
        "video": mp4_path,
        "prompt": build_anchor_prompt(frame_indices),
        "ground_truth": anchor_gt,
        "clip_json_path": json_path
    })

with open(phase_out, "w", encoding="utf-8") as f:
    for row in phase_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(anchor_out, "w", encoding="utf-8") as f:
    for row in anchor_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("saved phase task:", phase_out)
print("num phase samples:", len(phase_rows))

print("saved anchor task:", anchor_out)
print("num anchor samples:", len(anchor_rows))

if len(phase_rows) > 0:
    print("\n===== Phase task example =====")
    print(json.dumps(phase_rows[0], indent=2, ensure_ascii=False))

if len(anchor_rows) > 0:
    print("\n===== Anchor task example =====")
    print(json.dumps(anchor_rows[0], indent=2, ensure_ascii=False))

saved phase task: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/test_phase_task.jsonl
num phase samples: 3
saved anchor task: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/test_anchor_task.jsonl
num anchor samples: 3

===== Phase task example =====
{
  "video": "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/VID06_clip_02.mp4",
  "prompt": "<video>\nUse the following prior information and label mappings. These mappings are provided only to explain the meaning of each integer ID. In the final JSON output, you must use integer IDs only. Do not output category names, text labels, or descriptions in the final JSON. Phase ID mapping: 0=Preparation; 1=Calot triangle dissection; 2=Clipping and cutting; 3=Gallbladder dissection; 4=Gallbladder retraction; 5=Cleaning and coagulation; 6=Gallbladder packaging. The video has frames indexed from 0 to 22499. Return only a valid JSON object with exactly one key: 'phase_segments'

In [34]:
import json
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info


test_pairs_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/test_phase_task.jsonl"
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
pred_out = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/zeroshot_phase_predictions.jsonl"


dataset = load_dataset("json", data_files=test_pairs_path)["train"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

def predict_one(sample):
    prompt = sample["prompt"].strip()
    video_path = sample["video"]

    prompt = prompt.replace("<video>\n", "").replace("<video>", "").strip()

    if not video_path.startswith("file://"):
        video_path_for_msg = "file://" + video_path
    else:
        video_path_for_msg = video_path

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_path_for_msg,
                    "fps": 0.25,
                    "min_pixels": 256 * 256,
                    "max_pixels": 512 * 512
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        return_video_kwargs=True
    )

    fixed_video_kwargs = {}
    for k, v in video_kwargs.items():
        if isinstance(v, list) and len(v) == 1:
            fixed_video_kwargs[k] = v[0]
        else:
            fixed_video_kwargs[k] = v

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
        **fixed_video_kwargs
    )

    for k, v in inputs.items():
        if isinstance(v, torch.Tensor):
            inputs[k] = v.to(model.device)

    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False
        )

    input_token_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, input_token_len:]
    pred = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    return pred

rows = []
for i in range(len(dataset)):
    print(f"running phase task {i+1}/{len(dataset)}")
    sample = dataset[i]
    pred = predict_one(sample)

    rows.append({
        "video": sample["video"],
        "prompt": sample["prompt"],
        "ground_truth": sample["ground_truth"],
        "prediction": pred,
        "clip_json_path": sample["clip_json_path"]
    })

with open(pred_out, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("saved:", pred_out)

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

running phase task 1/3
running phase task 2/3
running phase task 3/3
saved: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/zeroshot_phase_predictions.jsonl


In [35]:
import json
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info


test_pairs_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/test_anchor_task.jsonl"
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
pred_out = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/zeroshot_anchor_predictions.jsonl"


dataset = load_dataset("json", data_files=test_pairs_path)["train"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

def predict_one(sample):
    prompt = sample["prompt"].strip()
    video_path = sample["video"]

    prompt = prompt.replace("<video>\n", "").replace("<video>", "").strip()

    if not video_path.startswith("file://"):
        video_path_for_msg = "file://" + video_path
    else:
        video_path_for_msg = video_path

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_path_for_msg,
                    "fps": 0.25,
                    "min_pixels": 256 * 256,
                    "max_pixels": 512 * 512
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        return_video_kwargs=True
    )

    fixed_video_kwargs = {}
    for k, v in video_kwargs.items():
        if isinstance(v, list) and len(v) == 1:
            fixed_video_kwargs[k] = v[0]
        else:
            fixed_video_kwargs[k] = v

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
        **fixed_video_kwargs
    )

    for k, v in inputs.items():
        if isinstance(v, torch.Tensor):
            inputs[k] = v.to(model.device)

    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=15000,
            do_sample=False
        )

    input_token_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, input_token_len:]
    pred = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    return pred

rows = []
for i in range(len(dataset)):
    print(f"running anchor task {i+1}/{len(dataset)}")
    sample = dataset[i]
    pred = predict_one(sample)

    rows.append({
        "video": sample["video"],
        "prompt": sample["prompt"],
        "ground_truth": sample["ground_truth"],
        "prediction": pred,
        "clip_json_path": sample["clip_json_path"]
    })

with open(pred_out, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("saved:", pred_out)

Generating train split: 0 examples [00:00, ? examples/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

running anchor task 1/3
running anchor task 2/3
running anchor task 3/3
saved: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/zeroshot_anchor_predictions.jsonl


# EVALUATION

## PHASE REC

* JSON parse success rate
* schema validity rate
* frame-wise phase accuracy
* frame-wise phase macro F1
* phase confusion matrix

## TOOLS DETECTION

A. Format

* JSON parse success rate
* schema validity rate

B. Phase

* accuracy
* macro F1
* confusion matrix

C. Detection / localization

* Hungarian matching
* precision
* recall
* F1
* bbox mean IoU
* per-instrument mean IoU
* bbox IoU bar chart

D. Matched-object classification

instrument

* accuracy
* per-class precision / recall / F1

operator

* accuracy
* per-class precision / recall / F1

intraoperative_track

* exact match accuracy
* macro/micro F1

intracorporeal_track

* exact match accuracy
* macro/micro F1

In [38]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


pred_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/zeroshot_phase_predictions.jsonl"
output_dir = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/eval_phase_task"


os.makedirs(output_dir, exist_ok=True)

phase_map = {
    0: "Preparation",
    1: "Calot triangle dissection",
    2: "Clipping and cutting",
    3: "Gallbladder dissection",
    4: "Gallbladder retraction",
    5: "Cleaning and coagulation",
    6: "Gallbladder packaging",
}

def extract_json_block(text):
    if not isinstance(text, str):
        return text
    text = text.strip()
    if "```json" in text:
        start = text.find("```json") + len("```json")
        end = text.rfind("```")
        if end > start:
            text = text[start:end].strip()
    elif "```" in text:
        start = text.find("```") + 3
        end = text.rfind("```")
        if end > start:
            text = text[start:end].strip()
    return text

def safe_parse_json(x):
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        return None
    try:
        return json.loads(extract_json_block(x))
    except Exception:
        return None

def schema_valid_phase(obj):
    if not isinstance(obj, dict):
        return False
    segs = obj.get("phase_segments")
    if not isinstance(segs, list):
        return False
    for seg in segs:
        if not isinstance(seg, dict):
            return False
        required = {"start_frame", "end_frame", "phase"}
        if not required.issubset(seg.keys()):
            return False
    return True

def segments_to_frame_labels(segments):

    if not isinstance(segments, list) or len(segments) == 0:
        return []

    max_end = max(int(seg["end_frame"]) for seg in segments)
    arr = [-1] * (max_end + 1)

    for seg in segments:
        s = int(seg["start_frame"])
        e = int(seg["end_frame"])
        p = int(seg["phase"])
        s = max(s, 0)
        e = max(e, s)
        for i in range(s, e + 1):
            if i < len(arr):
                arr[i] = p
    return arr

def save_confusion_matrix(cm, labels, title, out_path):
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45, ha="right")
    plt.yticks(ticks, labels)

    thresh = cm.max() / 2.0 if cm.size > 0 else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, str(cm[i, j]),
                ha="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.ylabel("Ground Truth")
    plt.xlabel("Prediction")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

with open(pred_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

num_samples = len(rows)
parse_ok = 0
schema_ok = 0

all_gt = []
all_pred = []

for row in rows:
    gt = safe_parse_json(row.get("ground_truth"))
    pred = safe_parse_json(row.get("prediction"))

    if pred is not None:
        parse_ok += 1
    if schema_valid_phase(pred):
        schema_ok += 1

    if gt is None or pred is None:
        continue
    if not schema_valid_phase(gt) or not schema_valid_phase(pred):
        continue

    gt_seq = segments_to_frame_labels(gt["phase_segments"])
    pred_seq = segments_to_frame_labels(pred["phase_segments"])

    n = min(len(gt_seq), len(pred_seq))
    if n == 0:
        continue

    gt_seq = gt_seq[:n]
    pred_seq = pred_seq[:n]

    for g, p in zip(gt_seq, pred_seq):
        if g != -1 and p != -1:
            all_gt.append(g)
            all_pred.append(p)

json_parse_success_rate = parse_ok / num_samples if num_samples > 0 else 0.0
schema_validity_rate = schema_ok / num_samples if num_samples > 0 else 0.0
phase_accuracy = accuracy_score(all_gt, all_pred) if len(all_gt) > 0 else 0.0
phase_macro_f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0) if len(all_gt) > 0 else 0.0

summary = {
    "json_parse_success_rate": json_parse_success_rate,
    "schema_validity_rate": schema_validity_rate,
    "framewise_phase_accuracy": phase_accuracy,
    "framewise_phase_macro_f1": phase_macro_f1,
}

with open(os.path.join(output_dir, "summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(summary)

if len(all_gt) > 0:
    labels = sorted(set(all_gt) | set(all_pred))
    cm = confusion_matrix(all_gt, all_pred, labels=labels)
    label_names = [phase_map.get(x, f"phase_{x}") for x in labels]

    save_confusion_matrix(
        cm,
        label_names,
        "Phase Task Confusion Matrix",
        os.path.join(output_dir, "phase_confusion_matrix.png")
    )

print("saved to:", output_dir)

{'json_parse_success_rate': 1.0, 'schema_validity_rate': 1.0, 'framewise_phase_accuracy': 0.9339265262972425, 'framewise_phase_macro_f1': 0.32194484936831874}
saved to: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/eval_phase_task


In [39]:
import os
import json
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

pred_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/zeroshot_anchor_predictions.jsonl"
output_dir = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/eval_anchor_task"
iou_threshold = 0.5


os.makedirs(output_dir, exist_ok=True)

instrument_map = {
    0: "grasper",
    1: "bipolar",
    2: "hook",
    3: "scissors",
    4: "clipper",
    5: "irrigator",
    6: "specimen-bag",
}

operator_map = {
    0: "null",
    1: "main-surgeon-left-hand (MSLH)",
    2: "assistant-surgeon-right-hand (ASRH)",
    3: "main-surgeon-right-hand (MSRH)",
}

phase_map = {
    0: "Preparation",
    1: "Calot triangle dissection",
    2: "Clipping and cutting",
    3: "Gallbladder dissection",
    4: "Gallbladder retraction",
    5: "Cleaning and coagulation",
    6: "Gallbladder packaging",
}


# -------------------------
# Utils
# -------------------------
def extract_json_block(text):
    if not isinstance(text, str):
        return text
    text = text.strip()
    if "```json" in text:
        start = text.find("```json") + len("```json")
        end = text.rfind("```")
        if end > start:
            text = text[start:end].strip()
    elif "```" in text:
        start = text.find("```") + 3
        end = text.rfind("```")
        if end > start:
            text = text[start:end].strip()
    return text


def safe_parse_json(x):
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        return None
    try:
        return json.loads(extract_json_block(x))
    except Exception:
        return None


def normalize_bbox(b):

    if b is None:
        return None

    # [[a,b,c,d]] -> [a,b,c,d]
    if isinstance(b, list) and len(b) == 1 and isinstance(b[0], list):
        b = b[0]

    if isinstance(b, list) and len(b) == 4 and all(isinstance(v, (int, float)) for v in b):
        return [float(x) for x in b]

    return None


def normalize_track(x):
    return x if isinstance(x, int) else None


def normalize_object(obj):
    if not isinstance(obj, dict):
        return {
            "instrument": None,
            "tool_bbox": None,
            "operator": None,
            "intraoperative_track": None,
            "intracorporeal_track": None,
        }

    instrument = obj.get("instrument")
    operator = obj.get("operator")

    return {
        "instrument": instrument if isinstance(instrument, int) else None,
        "tool_bbox": normalize_bbox(obj.get("tool_bbox")),
        "operator": operator if isinstance(operator, int) else None,
        "intraoperative_track": normalize_track(obj.get("intraoperative_track")),
        "intracorporeal_track": normalize_track(obj.get("intracorporeal_track")),
    }


def normalize_frame(frame):
    if not isinstance(frame, dict):
        return {"frame_index": None, "phase": None, "objects": []}

    frame_index = frame.get("frame_index")
    phase = frame.get("phase")
    objects = frame.get("objects", [])

    if not isinstance(objects, list):
        objects = []

    return {
        "frame_index": frame_index if isinstance(frame_index, int) else None,
        "phase": phase if isinstance(phase, int) else None,
        "objects": [normalize_object(o) for o in objects],
    }


def get_frames(obj):
    if not isinstance(obj, dict):
        return []
    frames = obj.get("frames", [])
    if not isinstance(frames, list):
        return []
    return [normalize_frame(f) for f in frames]


def schema_valid(obj):
    if not isinstance(obj, dict):
        return False

    frames = obj.get("frames")
    if not isinstance(frames, list):
        return False

    for frame in frames:
        if not isinstance(frame, dict):
            return False
        if "frame_index" not in frame or "phase" not in frame or "objects" not in frame:
            return False
        if not isinstance(frame["objects"], list):
            return False

        for ob in frame["objects"]:
            if not isinstance(ob, dict):
                return False
            required = {
                "instrument",
                "tool_bbox",
                "operator",
                "intraoperative_track",
                "intracorporeal_track",
            }
            if not required.issubset(ob.keys()):
                return False

    return True


def valid_bbox(b):
    return (
        isinstance(b, list)
        and len(b) == 4
        and all(isinstance(v, (int, float)) for v in b)
    )


def bbox_iou_xywh(box1, box2):
    if not valid_bbox(box1) or not valid_bbox(box2):
        return 0.0

    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    ax1, ay1, ax2, ay2 = x1, y1, x1 + w1, y1 + h1
    bx1, by1, bx2, by2 = x2, y2, x2 + w2, y2 + h2

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area1 = max(0.0, w1) * max(0.0, h1)
    area2 = max(0.0, w2) * max(0.0, h2)
    union = area1 + area2 - inter_area

    if union <= 0:
        return 0.0
    return inter_area / union


def hungarian_match(gt_objects, pred_objects, iou_thr=0.5):
    n_gt = len(gt_objects)
    n_pred = len(pred_objects)

    if n_gt == 0 and n_pred == 0:
        return [], [], []
    if n_gt == 0:
        return [], [], list(range(n_pred))
    if n_pred == 0:
        return [], list(range(n_gt)), []

    iou_mat = np.zeros((n_gt, n_pred), dtype=np.float32)
    for i, g in enumerate(gt_objects):
        for j, p in enumerate(pred_objects):
            iou_mat[i, j] = bbox_iou_xywh(g.get("tool_bbox"), p.get("tool_bbox"))

    cost = 1.0 - iou_mat
    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    matched_gt = set()
    matched_pred = set()

    for r, c in zip(row_ind, col_ind):
        iou = float(iou_mat[r, c])
        if iou >= iou_thr:
            matches.append((r, c, iou))
            matched_gt.add(r)
            matched_pred.add(c)

    unmatched_gt = [i for i in range(n_gt) if i not in matched_gt]
    unmatched_pred = [j for j in range(n_pred) if j not in matched_pred]

    return matches, unmatched_gt, unmatched_pred


def save_confusion_matrix(cm, labels, title, out_path):
    fig_w = max(8, 0.8 * len(labels))
    fig_h = max(6, 0.8 * len(labels))
    plt.figure(figsize=(fig_w, fig_h))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45, ha="right")
    plt.yticks(ticks, labels)

    thresh = cm.max() / 2.0 if cm.size > 0 else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, str(cm[i, j]),
                ha="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.ylabel("Ground Truth")
    plt.xlabel("Prediction")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()


def save_bar_chart(labels, values, title, ylabel, out_path):
    plt.figure(figsize=(max(10, 0.9 * len(labels)), 6))
    plt.bar(labels, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()


# -------------------------
# Load predictions
# -------------------------
with open(pred_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

num_samples = len(rows)

# -------------------------
# A. Format
# -------------------------
parse_ok = 0
schema_ok = 0

# -------------------------
# B. Phase
# -------------------------
phase_gt_all = []
phase_pred_all = []

# -------------------------
# C. Detection / localization
# -------------------------
tp = 0
fp = 0
fn = 0
iou_values = []
per_instrument_iou = defaultdict(list)

# -------------------------
# D. Matched-object classification
# -------------------------
instr_gt_all = []
instr_pred_all = []

op_gt_all = []
op_pred_all = []

intraop_gt_all = []
intraop_pred_all = []

intracorp_gt_all = []
intracorp_pred_all = []


# -------------------------
# Main loop
# -------------------------
for row in rows:
    gt = safe_parse_json(row.get("ground_truth"))
    pred = safe_parse_json(row.get("prediction"))

    gt_valid = schema_valid(gt)
    pred_valid = schema_valid(pred)

    if pred is not None:
        parse_ok += 1
    if pred_valid:
        schema_ok += 1

    if (not gt_valid) or (not pred_valid):
        continue

    gt_frames = get_frames(gt)
    pred_frames = get_frames(pred)

    gt_map = {f["frame_index"]: f for f in gt_frames if f["frame_index"] is not None}
    pred_map = {f["frame_index"]: f for f in pred_frames if f["frame_index"] is not None}

    common_frame_indices = sorted(set(gt_map.keys()) & set(pred_map.keys()))
    gt_only_indices = sorted(set(gt_map.keys()) - set(pred_map.keys()))
    pred_only_indices = sorted(set(pred_map.keys()) - set(gt_map.keys()))

    for fid in common_frame_indices:
        gt_f = gt_map[fid]
        pred_f = pred_map[fid]

        if gt_f["phase"] is not None and pred_f["phase"] is not None:
            phase_gt_all.append(gt_f["phase"])
            phase_pred_all.append(pred_f["phase"])

        gt_objs = gt_f["objects"]
        pred_objs = pred_f["objects"]

        matches, unmatched_gt, unmatched_pred = hungarian_match(
            gt_objs, pred_objs, iou_thr=iou_threshold
        )

        tp += len(matches)
        fp += len(unmatched_pred)
        fn += len(unmatched_gt)

        for gi, pi, iou in matches:
            g = gt_objs[gi]
            p = pred_objs[pi]

            iou_values.append(iou)

            if g["instrument"] is not None:
                per_instrument_iou[g["instrument"]].append(iou)

            if g["instrument"] is not None and p["instrument"] is not None:
                instr_gt_all.append(g["instrument"])
                instr_pred_all.append(p["instrument"])

            if g["operator"] is not None and p["operator"] is not None:
                op_gt_all.append(g["operator"])
                op_pred_all.append(p["operator"])

            if g["intraoperative_track"] is not None and p["intraoperative_track"] is not None:
                intraop_gt_all.append(g["intraoperative_track"])
                intraop_pred_all.append(p["intraoperative_track"])

            if g["intracorporeal_track"] is not None and p["intracorporeal_track"] is not None:
                intracorp_gt_all.append(g["intracorporeal_track"])
                intracorp_pred_all.append(p["intracorporeal_track"])

    for fid in gt_only_indices:
        fn += len(gt_map[fid]["objects"])

    for fid in pred_only_indices:
        fp += len(pred_map[fid]["objects"])


# -------------------------
# Compute metrics
# -------------------------
json_parse_success_rate = parse_ok / num_samples if num_samples > 0 else 0.0
schema_validity_rate = schema_ok / num_samples if num_samples > 0 else 0.0

phase_accuracy = accuracy_score(phase_gt_all, phase_pred_all) if len(phase_gt_all) > 0 else 0.0
phase_macro_f1 = f1_score(phase_gt_all, phase_pred_all, average="macro", zero_division=0) if len(phase_gt_all) > 0 else 0.0

detection_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
detection_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
detection_f1 = (
    2 * detection_precision * detection_recall / (detection_precision + detection_recall)
    if (detection_precision + detection_recall) > 0 else 0.0
)

bbox_mean_iou = float(np.mean(iou_values)) if len(iou_values) > 0 else 0.0

instrument_accuracy = accuracy_score(instr_gt_all, instr_pred_all) if len(instr_gt_all) > 0 else 0.0
instrument_macro_f1 = f1_score(instr_gt_all, instr_pred_all, average="macro", zero_division=0) if len(instr_gt_all) > 0 else 0.0
instrument_report = classification_report(
    instr_gt_all,
    instr_pred_all,
    labels=sorted(instrument_map.keys()),
    target_names=[instrument_map[k] for k in sorted(instrument_map.keys())],
    zero_division=0,
    output_dict=False
) if len(instr_gt_all) > 0 else "No matched objects."

operator_accuracy = accuracy_score(op_gt_all, op_pred_all) if len(op_gt_all) > 0 else 0.0
operator_macro_f1 = f1_score(op_gt_all, op_pred_all, average="macro", zero_division=0) if len(op_gt_all) > 0 else 0.0
operator_report = classification_report(
    op_gt_all,
    op_pred_all,
    labels=sorted(operator_map.keys()),
    target_names=[operator_map[k] for k in sorted(operator_map.keys())],
    zero_division=0,
    output_dict=False
) if len(op_gt_all) > 0 else "No matched objects."

intraoperative_track_exact_match_accuracy = accuracy_score(intraop_gt_all, intraop_pred_all) if len(intraop_gt_all) > 0 else 0.0
intraoperative_track_macro_f1 = f1_score(intraop_gt_all, intraop_pred_all, average="macro", zero_division=0) if len(intraop_gt_all) > 0 else 0.0
intraoperative_track_micro_f1 = f1_score(intraop_gt_all, intraop_pred_all, average="micro", zero_division=0) if len(intraop_gt_all) > 0 else 0.0

intracorporeal_track_exact_match_accuracy = accuracy_score(intracorp_gt_all, intracorp_pred_all) if len(intracorp_gt_all) > 0 else 0.0
intracorporeal_track_macro_f1 = f1_score(intracorp_gt_all, intracorp_pred_all, average="macro", zero_division=0) if len(intracorp_gt_all) > 0 else 0.0
intracorporeal_track_micro_f1 = f1_score(intracorp_gt_all, intracorp_pred_all, average="micro", zero_division=0) if len(intracorp_gt_all) > 0 else 0.0


# -------------------------
# Save summary
# -------------------------
summary = {
    "A_format": {
        "json_parse_success_rate": json_parse_success_rate,
        "schema_validity_rate": schema_validity_rate,
    },
    "B_phase": {
        "accuracy": phase_accuracy,
        "macro_f1": phase_macro_f1,
    },
    "C_detection_localization": {
        "iou_threshold_for_matching": iou_threshold,
        "precision": detection_precision,
        "recall": detection_recall,
        "f1": detection_f1,
        "bbox_mean_iou": bbox_mean_iou,
    },
    "D_matched_object_classification": {
        "instrument": {
            "accuracy": instrument_accuracy,
            "macro_f1": instrument_macro_f1,
        },
        "operator": {
            "accuracy": operator_accuracy,
            "macro_f1": operator_macro_f1,
        },
        "intraoperative_track": {
            "exact_match_accuracy": intraoperative_track_exact_match_accuracy,
            "macro_f1": intraoperative_track_macro_f1,
            "micro_f1": intraoperative_track_micro_f1,
        },
        "intracorporeal_track": {
            "exact_match_accuracy": intracorporeal_track_exact_match_accuracy,
            "macro_f1": intracorporeal_track_macro_f1,
            "micro_f1": intracorporeal_track_micro_f1,
        },
    },
}

with open(os.path.join(output_dir, "summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

with open(os.path.join(output_dir, "instrument_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(instrument_report)

with open(os.path.join(output_dir, "operator_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(operator_report)

if len(phase_gt_all) > 0:
    labels = sorted(set(phase_gt_all) | set(phase_pred_all))
    cm = confusion_matrix(phase_gt_all, phase_pred_all, labels=labels)
    label_names = [phase_map.get(x, f"phase_{x}") for x in labels]

    save_confusion_matrix(
        cm,
        label_names,
        "Anchor Task Phase Confusion Matrix",
        os.path.join(output_dir, "phase_confusion_matrix.png")
    )

inst_ids_sorted = sorted(per_instrument_iou.keys())
inst_names = [instrument_map.get(i, f"instrument_{i}") for i in inst_ids_sorted]
inst_mean_ious = [
    float(np.mean(per_instrument_iou[i])) if len(per_instrument_iou[i]) > 0 else 0.0
    for i in inst_ids_sorted
]

if len(inst_ids_sorted) > 0:
    save_bar_chart(
        inst_names,
        inst_mean_ious,
        "BBox Mean IoU by Instrument Class",
        "Mean IoU",
        os.path.join(output_dir, "bbox_iou_by_instrument.png")
    )

print("=" * 80)
print("A. FORMAT")
print("=" * 80)
print("JSON parse success rate:", round(json_parse_success_rate, 4))
print("Schema validity rate:", round(schema_validity_rate, 4))

print("\n" + "=" * 80)
print("B. PHASE")
print("=" * 80)
print("Accuracy:", round(phase_accuracy, 4))
print("Macro F1:", round(phase_macro_f1, 4))

print("\n" + "=" * 80)
print("C. DETECTION / LOCALIZATION")
print("=" * 80)
print("Hungarian matching IoU threshold:", iou_threshold)
print("Precision:", round(detection_precision, 4))
print("Recall:", round(detection_recall, 4))
print("F1:", round(detection_f1, 4))
print("BBox mean IoU:", round(bbox_mean_iou, 4))

print("\n" + "=" * 80)
print("D. MATCHED-OBJECT CLASSIFICATION")
print("=" * 80)

print("\nInstrument")
print("Accuracy:", round(instrument_accuracy, 4))
print("Macro F1:", round(instrument_macro_f1, 4))
print(instrument_report)

print("\nOperator")
print("Accuracy:", round(operator_accuracy, 4))
print("Macro F1:", round(operator_macro_f1, 4))
print(operator_report)

print("\nIntraoperative Track")
print("Exact match accuracy:", round(intraoperative_track_exact_match_accuracy, 4))
print("Macro F1:", round(intraoperative_track_macro_f1, 4))
print("Micro F1:", round(intraoperative_track_micro_f1, 4))

print("\nIntracorporeal Track")
print("Exact match accuracy:", round(intracorporeal_track_exact_match_accuracy, 4))
print("Macro F1:", round(intracorporeal_track_macro_f1, 4))
print("Micro F1:", round(intracorporeal_track_micro_f1, 4))

print("\nSaved to:", output_dir)
print("- summary_metrics.json")
print("- instrument_classification_report.txt")
print("- operator_classification_report.txt")
print("- phase_confusion_matrix.png")
print("- bbox_iou_by_instrument.png")

A. FORMAT
JSON parse success rate: 0.3333
Schema validity rate: 0.0

B. PHASE
Accuracy: 0.0
Macro F1: 0.0

C. DETECTION / LOCALIZATION
Hungarian matching IoU threshold: 0.5
Precision: 0.0
Recall: 0.0
F1: 0.0
BBox mean IoU: 0.0

D. MATCHED-OBJECT CLASSIFICATION

Instrument
Accuracy: 0.0
Macro F1: 0.0
No matched objects.

Operator
Accuracy: 0.0
Macro F1: 0.0
No matched objects.

Intraoperative Track
Exact match accuracy: 0.0
Macro F1: 0.0
Micro F1: 0.0

Intracorporeal Track
Exact match accuracy: 0.0
Macro F1: 0.0
Micro F1: 0.0

Saved to: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/eval_anchor_task
- summary_metrics.json
- instrument_classification_report.txt
- operator_classification_report.txt
- phase_confusion_matrix.png
- bbox_iou_by_instrument.png


# FINETUNE

## GT GEN

In [45]:
import os
import json

data_dir = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing"
out_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetune_long_video_10s.jsonl"
sample_stride_sec = 10


def round_bbox(bbox, ndigits=4):
    return [round(float(x), ndigits) for x in bbox]


def get_target_frame_indices(num_frames_in_clip, fps, stride_sec=10):
    frame_indices = []
    sec = 0
    while True:
        fid = int(round(sec * stride_sec * fps))
        if fid >= num_frames_in_clip:
            break
        frame_indices.append(fid)
        sec += 1
    return frame_indices


def build_one_video_training_row(mp4_path, json_path, sample, stride_sec=10):
    annotated_frame_ids = sorted(sample["annotated_frame_ids_local"])
    annotations_in_clip = sample["annotations_in_clip"]
    fps = float(sample["fps"])
    num_frames_in_clip = int(sample["num_frames_in_clip"])

    if len(annotated_frame_ids) == 0:
        return None

    target_frame_indices = get_target_frame_indices(
        num_frames_in_clip=num_frames_in_clip,
        fps=fps,
        stride_sec=stride_sec
    )

    phase_labels = []
    anchor_targets = []

    for target_fid in target_frame_indices:
        nearest_fid = min(annotated_frame_ids, key=lambda x: abs(x - target_fid))
        ann_list = annotations_in_clip[str(nearest_fid)]

        phase = ann_list[0]["phase"] if len(ann_list) > 0 else -1
        phase_labels.append(int(phase))

        objects = []
        for ann in ann_list:
            objects.append({
                "instrument": int(ann["instrument"]),
                "tool_bbox": round_bbox(ann["tool_bbox"]),
                "operator": int(ann["operator"]),
                "intraoperative_track": int(ann["intraoperative_track"]),
                "intracorporeal_track": int(ann["intracorporeal_track"]),
            })

        anchor_targets.append({
            "frame_index": int(target_fid),
            "phase": int(phase),
            "objects": objects
        })

    row = {
        "video": mp4_path,
        "clip_json_path": json_path,
        "fps": fps,
        "num_frames_in_clip": num_frames_in_clip,
        "sample_stride_sec": int(stride_sec),
        "frame_indices": [int(x) for x in target_frame_indices],
        "phase_labels": phase_labels,
        "anchor_targets": anchor_targets
    }
    return row


files = sorted(os.listdir(data_dir))
mp4_files = [f for f in files if f.endswith(".mp4") and "_clip_" in f]

rows = []

for mp4_name in mp4_files:
    base = os.path.splitext(mp4_name)[0]
    json_name = base + ".json"

    mp4_path = os.path.join(data_dir, mp4_name)
    json_path = os.path.join(data_dir, json_name)

    if not os.path.exists(json_path):
        print("[WARNING] missing json for:", mp4_name)
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        sample = json.load(f)

    if sample["num_annotated_frames"] == 0:
        print("[SKIP] no annotations:", json_name)
        continue

    row = build_one_video_training_row(
        mp4_path=mp4_path,
        json_path=json_path,
        sample=sample,
        stride_sec=sample_stride_sec
    )

    if row is not None:
        rows.append(row)

with open(out_path, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("saved:", out_path)
print("num videos:", len(rows))

if len(rows) > 0:
    print("\n===== Example =====")
    print(json.dumps(rows[0], indent=2, ensure_ascii=False))

saved: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetune_long_video_10s.jsonl
num videos: 3

===== Example =====
{
  "video": "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/VID06_clip_02.mp4",
  "clip_json_path": "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/VID06_clip_02.json",
  "fps": 25.0,
  "num_frames_in_clip": 22500,
  "sample_stride_sec": 10,
  "frame_indices": [
    0,
    250,
    500,
    750,
    1000,
    1250,
    1500,
    1750,
    2000,
    2250,
    2500,
    2750,
    3000,
    3250,
    3500,
    3750,
    4000,
    4250,
    4500,
    4750,
    5000,
    5250,
    5500,
    5750,
    6000,
    6250,
    6500,
    6750,
    7000,
    7250,
    7500,
    7750,
    8000,
    8250,
    8500,
    8750,
    9000,
    9250,
    9500,
    9750,
    10000,
    10250,
    10500,
    10750,
    11000,
    11250,
    11500,
    11750,
    12000,
    12250,
    12500,
    12750,
    13000,
    1325

## PHASE RECOGNITION

In [10]:
import os
import json
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2_5_VLModel
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import time


train_jsonl = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/training/finetune_long_video_10s.jsonl"

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
save_dir = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/taskA_qwen25vl_lora"

num_epochs = 15
lr = 5e-6
num_phase_classes = 7
batch_size = 1
max_frames_per_video = 2
device = "cuda" if torch.cuda.is_available() else "cpu"


lora_r = 8
lora_alpha = 16
lora_dropout = 0.05


grad_clip_norm = 0.5
adam_eps = 1e-6
# =========================

os.makedirs(save_dir, exist_ok=True)
torch.autograd.set_detect_anomaly(True)


# -------------------------
# Utils
# -------------------------
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def can_open_video(video_path):
    if not os.path.exists(video_path):
        return False
    cap = cv2.VideoCapture(video_path)
    ok = cap.isOpened()
    cap.release()
    return ok




def read_frames_rgb(video_path, frame_indices, max_retries=3, sleep_sec=1.0):
    last_error = None

    for attempt in range(max_retries):
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            last_error = RuntimeError(f"Failed to open video: {video_path} | attempt={attempt+1}")
            cap.release()
            time.sleep(sleep_sec)
            continue

        frames = []
        ok_all = True

        for frame_index in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, frame = cap.read()
            if not ok or frame is None:
                ok_all = False
                last_error = RuntimeError(
                    f"Failed to read frame {frame_index} from {video_path} | attempt={attempt+1}"
                )
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        if ok_all:
            return frames

        time.sleep(sleep_sec)

    raise last_error


# -------------------------
# Dataset
# -------------------------
class TaskADataset(Dataset):
    def __init__(self, jsonl_path, max_frames_per_video=None, num_phase_classes=7):
        raw_rows = load_jsonl(jsonl_path)
        self.rows = []
        self.max_frames_per_video = max_frames_per_video
        self.num_phase_classes = num_phase_classes

        for row in raw_rows:
            video_path = row["video"]

            if not can_open_video(video_path):
                print("[SKIP] bad video:", video_path)
                continue

            frame_indices = row["frame_indices"]
            phase_labels = row["phase_labels"]

            if len(frame_indices) != len(phase_labels):
                print("[SKIP] length mismatch:", video_path)
                continue

            valid_count = sum(
                1 for x in phase_labels
                if isinstance(x, int) and 0 <= x < self.num_phase_classes
            )
            if valid_count == 0:
                print("[SKIP] no valid phase labels:", video_path)
                continue

            self.rows.append(row)

        print("num usable rows:", len(self.rows))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        video_path = row["video"]
        frame_indices = row["frame_indices"]
        phase_labels = row["phase_labels"]

        if self.max_frames_per_video is not None:
            frame_indices = frame_indices[:self.max_frames_per_video]
            phase_labels = phase_labels[:self.max_frames_per_video]

        frames = read_frames_rgb(video_path, frame_indices)

        return {
            "video_path": video_path,
            "frames": frames,
            "frame_indices": frame_indices,
            "phase_labels": phase_labels,
        }


def collate_fn(batch):
    assert len(batch) == 1
    return batch[0]


# -------------------------
# Model
# -------------------------
class TaskAQwen25VLLoRA(nn.Module):
    def __init__(
        self,
        model_name,
        num_phase_classes=7,
        lora_r=8,
        lora_alpha=16,
        lora_dropout=0.05
    ):
        super().__init__()

        self.processor = AutoProcessor.from_pretrained(model_name)

        backbone = Qwen2_5_VLModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        )
        self.backbone = get_peft_model(backbone, lora_config)

        hidden_size = self.backbone.config.text_config.hidden_size

        self.temporal = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.feat_norm = nn.LayerNorm(hidden_size)
        self.phase_head = nn.Linear(hidden_size, num_phase_classes)

        self.backbone.print_trainable_parameters()

    def encode_one_frame(self, frame_rgb):
        device = next(self.parameters()).device

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": frame_rgb},
                    {"type": "text", "text": "phase classification"}
                ]
            }
        ]

        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors="pt"
        )

        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(device)

        outputs = self.backbone(**inputs, return_dict=True)
        feat = outputs.last_hidden_state.mean(dim=1)[0]

        if torch.isnan(feat).any() or torch.isinf(feat).any():
            raise ValueError("feature has NaN/Inf inside encode_one_frame")

        return feat.float()

    def forward(self, frames):
        feats = []
        for frame_rgb in frames:
            feat = self.encode_one_frame(frame_rgb)
            feats.append(feat)

        feats = torch.stack(feats, dim=0).unsqueeze(0)   # [1, T, D]
        feats = self.feat_norm(feats)

        if torch.isnan(feats).any() or torch.isinf(feats).any():
            raise ValueError("feats has NaN/Inf after feat_norm")
        temporal_out, _ = self.temporal(feats)
        phase_logits = self.phase_head(temporal_out)            # [1, T, 7]

        if torch.isnan(phase_logits).any() or torch.isinf(phase_logits).any():
            raise ValueError("phase_logits has NaN/Inf")

        return phase_logits


# -------------------------
# Loss
# -------------------------
def compute_loss(phase_logits, phase_labels, num_phase_classes=7):
    logits = phase_logits[0]  # [T, 7]

    valid_indices = [
        i for i, x in enumerate(phase_labels)
        if isinstance(x, int) and 0 <= x < num_phase_classes
    ]

    if len(valid_indices) == 0:
        return None

    valid_logits = logits[valid_indices]
    valid_labels = torch.tensor(
        [phase_labels[i] for i in valid_indices],
        dtype=torch.long,
        device=logits.device
    )

    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(valid_logits, valid_labels)
    return loss


# -------------------------
# Train
# -------------------------
def train():
    train_dataset = TaskADataset(
        train_jsonl,
        max_frames_per_video=max_frames_per_video,
        num_phase_classes=num_phase_classes
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn
    )

    model = TaskAQwen25VLLoRA(
        model_name=model_name,
        num_phase_classes=num_phase_classes,
        lora_r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout
    ).to(device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        eps=adam_eps
    )

    global_step = 0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        steps = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for sample in pbar:
            global_step += 1

            try:
                phase_logits = model(sample["frames"])
            except Exception as e:
                print("\n[FORWARD FAIL]")
                print("video:", sample["video_path"])
                print("frame_indices:", sample["frame_indices"])
                raise e

            if torch.isnan(phase_logits).any() or torch.isinf(phase_logits).any():
                print("\n[BAD LOGITS]")
                print("video:", sample["video_path"])
                print("frame_indices:", sample["frame_indices"])
                print("phase_labels:", sample["phase_labels"])
                raise ValueError("phase_logits invalid")

            loss = compute_loss(
                phase_logits,
                sample["phase_labels"],
                num_phase_classes=num_phase_classes
            )

            if loss is None:
                print("\n[SKIP] no valid labels:", sample["video_path"])
                continue

            if torch.isnan(loss) or torch.isinf(loss):
                print("\n[BAD LOSS]")
                print("video:", sample["video_path"])
                print("phase_labels:", sample["phase_labels"])
                print("logits min/max:", phase_logits.min().item(), phase_logits.max().item())
                raise ValueError("loss invalid")

            optimizer.zero_grad()
            loss.backward()

            bad_grad = False
            for name, p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    if torch.isnan(p.grad).any() or torch.isinf(p.grad).any():
                        print("\n[BAD GRAD]", name)
                        print("video:", sample["video_path"])
                        bad_grad = True
                        break
            if bad_grad:
                raise ValueError("gradient invalid")

            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=grad_clip_norm)
            optimizer.step()

            bad_param = False
            for name, p in model.named_parameters():
                if p.requires_grad:
                    if torch.isnan(p.data).any() or torch.isinf(p.data).any():
                        print("\n[BAD PARAM AFTER STEP]", name)
                        print("video:", sample["video_path"])
                        bad_param = True
                        break
            if bad_param:
                raise ValueError("parameter invalid after step")

            running_loss += loss.item()
            steps += 1

            pbar.set_postfix(
                loss=f"{running_loss / max(steps,1):.4f}",
                last=f"{loss.item():.4f}",
                step=global_step
            )

        train_loss = running_loss / max(steps, 1)

        print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.4f}")

    ckpt = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
    }

    #torch.save(ckpt, os.path.join(save_dir, f"epoch_{epoch+1}.pt"))
    torch.save(ckpt, os.path.join(save_dir, "last.pt"))

    print("Training done.")
    print("Saved to:", save_dir)


if __name__ == "__main__":
    train()

num usable rows: 17


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 3,758,309,376 || trainable%: 0.0981


Epoch 1/15:   0%|          | 0/17 [00:00<?, ?it/s]

...

Epoch 13/15 | train_loss=1.3453


Epoch 14/15:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 14/15 | train_loss=1.3272


Epoch 15/15:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 15/15 | train_loss=1.3002
Training done.
Saved to: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/taskA_qwen25vl_lora


In [25]:
import os
import json
import cv2
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2_5_VLModel
from peft import LoraConfig, get_peft_model


test_jsonl = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetune_long_video_10s.jsonl"

ckpt_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/taskA_qwen25vl_lora/last.pt"

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
pred_out = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetuned_phase_predictions.jsonl"

num_phase_classes = 7
device = "cuda" if torch.cuda.is_available() else "cpu"

max_frames_per_video = 90
lora_r = 8
lora_alpha = 16
lora_dropout = 0.05



# def read_frames_rgb(video_path, frame_indices, max_retries=3, sleep_sec=1.0):
#     last_error = None

#     for attempt in range(max_retries):
#         cap = cv2.VideoCapture(video_path)

#         if not cap.isOpened():
#             last_error = RuntimeError(f"Failed to open video: {video_path} | attempt={attempt+1}")
#             cap.release()
#             time.sleep(sleep_sec)
#             continue

#         frames = []
#         ok_all = True

#         for frame_index in frame_indices:
#             cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
#             ok, frame = cap.read()
#             if not ok or frame is None:
#                 ok_all = False
#                 last_error = RuntimeError(
#                     f"Failed to read frame {frame_index} from {video_path} | attempt={attempt+1}"
#                 )
#                 break
#             frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             frames.append(frame)

#         cap.release()

#         if ok_all:
#             return frames

#         time.sleep(sleep_sec)

#     raise last_error

def sampled_phases_to_segments(frame_indices, pred_phase_ids, num_frames_in_clip):
    if len(frame_indices) == 0:
        return {"phase_segments": []}

    segments = []
    start_frame = int(frame_indices[0])
    current_phase = int(pred_phase_ids[0])

    for i in range(1, len(frame_indices)):
        fid = int(frame_indices[i])
        phase = int(pred_phase_ids[i])

        if phase != current_phase:
            segments.append({
                "start_frame": int(start_frame),
                "end_frame": int(fid - 1),
                "phase": int(current_phase)
            })
            start_frame = fid
            current_phase = phase

    segments.append({
        "start_frame": int(start_frame),
        "end_frame": int(num_frames_in_clip - 1),
        "phase": int(current_phase)
    })

    return {"phase_segments": segments}


class TaskATestDataset(Dataset):
    def __init__(self, jsonl_path, max_frames_per_video=None):
        self.rows = load_jsonl(jsonl_path)
        self.max_frames_per_video = max_frames_per_video

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        video_path = row["video"]
        frame_indices = row["frame_indices"]
        phase_labels = row["phase_labels"]

        if self.max_frames_per_video is not None:
            frame_indices = frame_indices[:self.max_frames_per_video]
            phase_labels = phase_labels[:self.max_frames_per_video]

        frames = read_frames_rgb(video_path, frame_indices)

        return {
            "video": video_path,
            "frames": frames,
            "frame_indices": frame_indices,
            "phase_labels": phase_labels,
            "clip_json_path": row.get("clip_json_path", None),
            "num_frames_in_clip": row.get("num_frames_in_clip", None),
        }



@torch.no_grad()
def run_inference():
    dataset = TaskATestDataset(test_jsonl, max_frames_per_video=max_frames_per_video)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

    model = TaskAQwen25VLLoRA(
        model_name=model_name,
        num_phase_classes=num_phase_classes,
        lora_r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()

    rows = []

    for i, sample in enumerate(loader):
        print(f"running phase inference {i+1}/{len(dataset)}")

        phase_logits = model(sample["frames"])
        pred_phase_ids = phase_logits.argmax(dim=-1)[0].detach().cpu().tolist()

        prediction = sampled_phases_to_segments(
            frame_indices=sample["frame_indices"],
            pred_phase_ids=pred_phase_ids,
            num_frames_in_clip=sample["num_frames_in_clip"]
        )

        ground_truth = sampled_phases_to_segments(
            frame_indices=sample["frame_indices"],
            pred_phase_ids=sample["phase_labels"],
            num_frames_in_clip=sample["num_frames_in_clip"]
        )

        rows.append({
            "video": sample["video"],
            "ground_truth": ground_truth,
            "prediction": prediction,
            "clip_json_path": sample["clip_json_path"]
        })

    with open(pred_out, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("saved:", pred_out)


if __name__ == "__main__":
    run_inference()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

running phase inference 1/3
running phase inference 2/3
running phase inference 3/3
saved: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetuned_phase_predictions.jsonl


In [26]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


pred_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetuned_phase_predictions.jsonl"
output_dir = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/eval_finetuned_phase_task"


os.makedirs(output_dir, exist_ok=True)



with open(pred_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

all_gt = []
all_pred = []

for row in rows:
    gt = row["ground_truth"]
    pred = row["prediction"]

    gt_segments = gt["phase_segments"]
    pred_segments = pred["phase_segments"]

    gt_seq = segments_to_frame_labels(gt_segments)
    pred_seq = segments_to_frame_labels(pred_segments)

    n = min(len(gt_seq), len(pred_seq))
    if n == 0:
        continue

    gt_seq = gt_seq[:n]
    pred_seq = pred_seq[:n]

    for g, p in zip(gt_seq, pred_seq):
        if g != -1 and p != -1:
            all_gt.append(g)
            all_pred.append(p)

phase_accuracy = accuracy_score(all_gt, all_pred) if len(all_gt) > 0 else 0.0
phase_macro_f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0) if len(all_gt) > 0 else 0.0

summary = {
    "framewise_phase_accuracy": phase_accuracy,
    "framewise_phase_macro_f1": phase_macro_f1,
}

with open(os.path.join(output_dir, "summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(summary)

if len(all_gt) > 0:
    labels = sorted(set(all_gt) | set(all_pred))
    cm = confusion_matrix(all_gt, all_pred, labels=labels)
    label_names = [phase_map.get(x, f"phase_{x}") for x in labels]

    save_confusion_matrix(
        cm,
        label_names,
        "Finetuned Phase Confusion Matrix",
        os.path.join(output_dir, "phase_confusion_matrix.png")
    )

print("saved to:", output_dir)

{'framewise_phase_accuracy': 0.15925925925925927, 'framewise_phase_macro_f1': 0.056514913657770796}
saved to: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/eval_finetuned_phase_task


## TOOL DETECTION

In [16]:
import os
import json
import time
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2_5_VLModel
from peft import LoraConfig, get_peft_model
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm


train_jsonl = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/training/finetune_long_video_10s.jsonl"
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
save_dir = "/content/drive/MyDrive/CholecTrack20/taskB_qwen25vl_lora"

num_epochs = 20
lr = 5e-6
batch_size = 1
max_frames_per_video = 1
device = "cuda" if torch.cuda.is_available() else "cpu"

num_phase_classes = 7
num_instrument_classes = 7
num_operator_classes = 4
num_slots = 3

# LoRA
lora_r = 4
lora_alpha = 8
lora_dropout = 0.05

# loss weight
lambda_phase = 1.0
lambda_obj = 1.0
lambda_instr = 1.0
lambda_op = 1.0
lambda_bbox_l1 = 2.0
lambda_bbox_iou = 2.0
lambda_intraop = 1.0
lambda_intracorp = 1.0

# matching cost weight
match_cost_bbox = 2.0
match_cost_instr = 1.0
match_cost_op = 1.0
match_cost_intraop = 0.5
match_cost_intracorp = 0.5

# stablize
grad_clip_norm = 0.5
adam_eps = 1e-6
# =========================

os.makedirs(save_dir, exist_ok=True)
torch.autograd.set_detect_anomaly(True)


# -------------------------
# Utils
# -------------------------
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def can_open_video(video_path):
    if not os.path.exists(video_path):
        return False
    cap = cv2.VideoCapture(video_path)
    ok = cap.isOpened()
    cap.release()
    return ok


def read_frames_rgb(video_path, frame_indices, max_retries=3, sleep_sec=1.0):
    last_error = None

    for attempt in range(max_retries):
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            last_error = RuntimeError(f"Failed to open video: {video_path} | attempt={attempt+1}")
            cap.release()
            time.sleep(sleep_sec)
            continue

        frames = []
        ok_all = True

        for frame_index in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, frame = cap.read()
            if not ok or frame is None:
                ok_all = False
                last_error = RuntimeError(
                    f"Failed to read frame {frame_index} from {video_path} | attempt={attempt+1}"
                )
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        if ok_all:
            return frames

        time.sleep(sleep_sec)

    raise last_error


def infer_track_vocab_sizes(rows):
    max_intraop = 0
    max_intracorp = 0
    for row in rows:
        for frame in row["anchor_targets"]:
            for obj in frame["objects"]:
                max_intraop = max(max_intraop, int(obj["intraoperative_track"]))
                max_intracorp = max(max_intracorp, int(obj["intracorporeal_track"]))
    return max_intraop + 1, max_intracorp + 1


def bbox_xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]


def box_area_xyxy(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_iou_xywh(box1, box2):
    if box1 is None or box2 is None:
        return 0.0
    ax1, ay1, ax2, ay2 = bbox_xywh_to_xyxy(box1)
    bx1, by1, bx2, by2 = bbox_xywh_to_xyxy(box2)

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = box_area_xyxy([ax1, ay1, ax2, ay2])
    area_b = box_area_xyxy([bx1, by1, bx2, by2])
    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0
    return inter_area / union


def bbox_iou_xywh_torch(box1, box2):
    """
    box1, box2: [4] in xywh
    """
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    ax1, ay1, ax2, ay2 = x1, y1, x1 + w1, y1 + h1
    bx1, by1, bx2, by2 = x2, y2, x2 + w2, y2 + h2

    inter_x1 = torch.max(ax1, bx1)
    inter_y1 = torch.max(ay1, by1)
    inter_x2 = torch.min(ax2, bx2)
    inter_y2 = torch.min(ay2, by2)

    inter_w = torch.clamp(inter_x2 - inter_x1, min=0.0)
    inter_h = torch.clamp(inter_y2 - inter_y1, min=0.0)
    inter_area = inter_w * inter_h

    area1 = torch.clamp(w1, min=0.0) * torch.clamp(h1, min=0.0)
    area2 = torch.clamp(w2, min=0.0) * torch.clamp(h2, min=0.0)
    union = area1 + area2 - inter_area

    iou = inter_area / (union + 1e-8)
    return iou


# -------------------------
# Dataset
# -------------------------
class TaskBDataset(Dataset):
    def __init__(self, jsonl_path, max_frames_per_video=None):
        raw_rows = load_jsonl(jsonl_path)
        self.rows = []
        self.max_frames_per_video = max_frames_per_video

        for row in raw_rows:
            video_path = row["video"]

            if not can_open_video(video_path):
                print("[SKIP] bad video:", video_path)
                continue

            frame_indices = row["frame_indices"]
            anchor_targets = row["anchor_targets"]

            if len(frame_indices) != len(anchor_targets):
                print("[SKIP] length mismatch:", video_path)
                continue

            self.rows.append(row)

        print("num usable rows:", len(self.rows))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        video_path = row["video"]
        frame_indices = row["frame_indices"]
        anchor_targets = row["anchor_targets"]

        if self.max_frames_per_video is not None:
            frame_indices = frame_indices[:self.max_frames_per_video]
            anchor_targets = anchor_targets[:self.max_frames_per_video]

        frames = read_frames_rgb(video_path, frame_indices)

        return {
            "video_path": video_path,
            "frames": frames,                   # len=T
            "frame_indices": frame_indices,     # len=T
            "anchor_targets": anchor_targets,   # len=T
        }


def collate_fn(batch):
    assert len(batch) == 1
    return batch[0]


# -------------------------
# Model
# -------------------------
class TaskBQwen25VLLoRA(nn.Module):
    def __init__(
        self,
        model_name,
        num_phase_classes,
        num_instrument_classes,
        num_operator_classes,
        num_intraop_classes,
        num_intracorp_classes,
        num_slots=3,
        lora_r=4,
        lora_alpha=8,
        lora_dropout=0.05,
    ):
        super().__init__()

        self.processor = AutoProcessor.from_pretrained(model_name)

        backbone = Qwen2_5_VLModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            target_modules=["q_proj", "v_proj"],
        )
        self.backbone = get_peft_model(backbone, lora_config)

        hidden_size = self.backbone.config.text_config.hidden_size
        self.hidden_size = hidden_size
        self.num_slots = num_slots
        self.num_instrument_classes = num_instrument_classes
        self.num_operator_classes = num_operator_classes
        self.num_intraop_classes = num_intraop_classes
        self.num_intracorp_classes = num_intracorp_classes

        self.feat_norm = nn.LayerNorm(hidden_size)

        self.temporal = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.phase_head = nn.Linear(hidden_size, num_phase_classes)
        self.obj_head = nn.Linear(hidden_size, num_slots)
        self.instr_head = nn.Linear(hidden_size, num_slots * num_instrument_classes)
        self.op_head = nn.Linear(hidden_size, num_slots * num_operator_classes)
        self.bbox_head = nn.Linear(hidden_size, num_slots * 4)
        self.intraop_head = nn.Linear(hidden_size, num_slots * num_intraop_classes)
        self.intracorp_head = nn.Linear(hidden_size, num_slots * num_intracorp_classes)

        self.backbone.print_trainable_parameters()

    def encode_one_frame(self, frame_rgb):
        device = next(self.parameters()).device

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": frame_rgb},
                    {"type": "text", "text": "surgical tool and phase understanding"}
                ]
            }
        ]

        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors="pt"
        )

        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(device)

        outputs = self.backbone(**inputs, return_dict=True)
        feat = outputs.last_hidden_state.mean(dim=1)[0]

        if torch.isnan(feat).any() or torch.isinf(feat).any():
            raise ValueError("feature has NaN/Inf inside encode_one_frame")

        return feat.float()

    def forward(self, frames):
        feats = []
        for frame_rgb in frames:
            feat = self.encode_one_frame(frame_rgb)
            feats.append(feat)

        feats = torch.stack(feats, dim=0).unsqueeze(0)   # [1, T, D]
        feats = self.feat_norm(feats)

        if torch.isnan(feats).any() or torch.isinf(feats).any():
            raise ValueError("feats has NaN/Inf after feat_norm")

        temporal_out, _ = self.temporal(feats)           # [1, T, D]

        phase_logits = self.phase_head(temporal_out)     # [1, T, P]
        obj_logits = self.obj_head(temporal_out)         # [1, T, K]

        t = temporal_out.shape[1]
        k = self.num_slots

        instr_logits = self.instr_head(temporal_out).view(1, t, k, self.num_instrument_classes)
        op_logits = self.op_head(temporal_out).view(1, t, k, self.num_operator_classes)
        bbox_pred = self.bbox_head(temporal_out).view(1, t, k, 4).sigmoid()
        intraop_logits = self.intraop_head(temporal_out).view(1, t, k, self.num_intraop_classes)
        intracorp_logits = self.intracorp_head(temporal_out).view(1, t, k, self.num_intracorp_classes)

        return {
            "phase_logits": phase_logits,
            "obj_logits": obj_logits,
            "instr_logits": instr_logits,
            "op_logits": op_logits,
            "bbox_pred": bbox_pred,
            "intraop_logits": intraop_logits,
            "intracorp_logits": intracorp_logits,
        }


# -------------------------
# Matching
# -------------------------
def build_matching_cost_for_one_t(
    instr_logits_t,      # [K, C_instr]
    op_logits_t,         # [K, C_op]
    bbox_pred_t,         # [K, 4]
    intraop_logits_t,    # [K, C_intraop]
    intracorp_logits_t,  # [K, C_intracorp]
    gt_objects           # list[dict], len=M
):
    k = instr_logits_t.shape[0]
    m = len(gt_objects)

    if m == 0:
        return None

    cost = torch.zeros((k, m), dtype=torch.float32, device=instr_logits_t.device)

    instr_prob = F.softmax(instr_logits_t, dim=-1)
    op_prob = F.softmax(op_logits_t, dim=-1)
    intraop_prob = F.softmax(intraop_logits_t, dim=-1)
    intracorp_prob = F.softmax(intracorp_logits_t, dim=-1)

    for j, gt in enumerate(gt_objects):
        gt_instr = int(gt["instrument"])
        gt_op = int(gt["operator"])
        gt_bbox = gt["tool_bbox"]
        gt_intraop = int(gt["intraoperative_track"])
        gt_intracorp = int(gt["intracorporeal_track"])

        for i in range(k):
            p_bbox = bbox_pred_t[i].detach().cpu().tolist()
            bbox_l1 = sum(abs(float(a) - float(b)) for a, b in zip(p_bbox, gt_bbox))
            instr_cost = -float(instr_prob[i, gt_instr].item())
            op_cost = -float(op_prob[i, gt_op].item())
            intraop_cost = -float(intraop_prob[i, gt_intraop].item())
            intracorp_cost = -float(intracorp_prob[i, gt_intracorp].item())

            total = (
                match_cost_bbox * bbox_l1
                + match_cost_instr * instr_cost
                + match_cost_op * op_cost
                + match_cost_intraop * intraop_cost
                + match_cost_intracorp * intracorp_cost
            )
            cost[i, j] = total

    return cost


def hungarian_match_one_t(
    instr_logits_t,
    op_logits_t,
    bbox_pred_t,
    intraop_logits_t,
    intracorp_logits_t,
    gt_objects
):
    k = instr_logits_t.shape[0]
    m = len(gt_objects)

    if m == 0:
        return [], list(range(k)), []

    cost = build_matching_cost_for_one_t(
        instr_logits_t,
        op_logits_t,
        bbox_pred_t,
        intraop_logits_t,
        intracorp_logits_t,
        gt_objects
    )

    row_ind, col_ind = linear_sum_assignment(cost.detach().cpu().numpy())

    matches = list(zip(row_ind.tolist(), col_ind.tolist()))
    matched_pred = set(r for r, _ in matches)
    unmatched_pred = [i for i in range(k) if i not in matched_pred]

    return matches, unmatched_pred, []


# -------------------------
# Loss
# -------------------------
def compute_taskb_loss(outputs, anchor_targets):
    phase_logits = outputs["phase_logits"][0]          # [T, P]
    obj_logits = outputs["obj_logits"][0]              # [T, K]
    instr_logits = outputs["instr_logits"][0]          # [T, K, C_instr]
    op_logits = outputs["op_logits"][0]                # [T, K, C_op]
    bbox_pred = outputs["bbox_pred"][0]                # [T, K, 4]
    intraop_logits = outputs["intraop_logits"][0]      # [T, K, C_intraop]
    intracorp_logits = outputs["intracorp_logits"][0]  # [T, K, C_intracorp]

    device = phase_logits.device
    t_len = len(anchor_targets)
    k = obj_logits.shape[1]

    # 1) phase loss
    phase_gt = torch.tensor(
        [int(x["phase"]) for x in anchor_targets],
        dtype=torch.long,
        device=device
    )
    loss_phase = F.cross_entropy(phase_logits, phase_gt)

    # 2) object/attr/bbox losses
    loss_obj = torch.tensor(0.0, device=device)
    loss_instr = torch.tensor(0.0, device=device)
    loss_op = torch.tensor(0.0, device=device)
    loss_bbox = torch.tensor(0.0, device=device)
    loss_intraop = torch.tensor(0.0, device=device)
    loss_intracorp = torch.tensor(0.0, device=device)

    count_obj_steps = 0
    count_matched = 0

    for t in range(t_len):
        gt_objects = anchor_targets[t]["objects"]

        matches, unmatched_pred, _ = hungarian_match_one_t(
            instr_logits[t],
            op_logits[t],
            bbox_pred[t],
            intraop_logits[t],
            intracorp_logits[t],
            gt_objects
        )

        # objectness GT
        obj_gt_t = torch.zeros((k,), dtype=torch.float32, device=device)
        for pred_idx, gt_idx in matches:
            obj_gt_t[pred_idx] = 1.0

        loss_obj = loss_obj + F.binary_cross_entropy_with_logits(
            obj_logits[t], obj_gt_t
        )
        count_obj_steps += 1

        # matched pairs
        for pred_idx, gt_idx in matches:
            gt = gt_objects[gt_idx]

            gt_instr = torch.tensor([int(gt["instrument"])], dtype=torch.long, device=device)
            gt_op = torch.tensor([int(gt["operator"])], dtype=torch.long, device=device)
            gt_intraop = torch.tensor([int(gt["intraoperative_track"])], dtype=torch.long, device=device)
            gt_intracorp = torch.tensor([int(gt["intracorporeal_track"])], dtype=torch.long, device=device)
            gt_bbox = torch.tensor(gt["tool_bbox"], dtype=torch.float32, device=device)

            loss_instr = loss_instr + F.cross_entropy(
                instr_logits[t, pred_idx].unsqueeze(0), gt_instr
            )
            loss_op = loss_op + F.cross_entropy(
                op_logits[t, pred_idx].unsqueeze(0), gt_op
            )
            loss_intraop = loss_intraop + F.cross_entropy(
                intraop_logits[t, pred_idx].unsqueeze(0), gt_intraop
            )
            loss_intracorp = loss_intracorp + F.cross_entropy(
                intracorp_logits[t, pred_idx].unsqueeze(0), gt_intracorp
            )

            pred_box = bbox_pred[t, pred_idx]
            l1 = F.l1_loss(pred_box, gt_bbox)
            iou = bbox_iou_xywh_torch(pred_box, gt_bbox)
            bbox_loss = lambda_bbox_l1 * l1 + lambda_bbox_iou * (1.0 - iou)
            loss_bbox = loss_bbox + bbox_loss

            count_matched += 1

    if count_obj_steps > 0:
        loss_obj = loss_obj / count_obj_steps

    if count_matched > 0:
        loss_instr = loss_instr / count_matched
        loss_op = loss_op / count_matched
        loss_bbox = loss_bbox / count_matched
        loss_intraop = loss_intraop / count_matched
        loss_intracorp = loss_intracorp / count_matched
    else:
        loss_instr = torch.tensor(0.0, device=device)
        loss_op = torch.tensor(0.0, device=device)
        loss_bbox = torch.tensor(0.0, device=device)
        loss_intraop = torch.tensor(0.0, device=device)
        loss_intracorp = torch.tensor(0.0, device=device)

    total_loss = (
        lambda_phase * loss_phase
        + lambda_obj * loss_obj
        + lambda_instr * loss_instr
        + lambda_op * loss_op
        + loss_bbox
        + lambda_intraop * loss_intraop
        + lambda_intracorp * loss_intracorp
    )

    loss_dict = {
        "total": total_loss.detach().item(),
        "phase": loss_phase.detach().item(),
        "obj": loss_obj.detach().item(),
        "instr": loss_instr.detach().item(),
        "op": loss_op.detach().item(),
        "bbox": loss_bbox.detach().item(),
        "intraop": loss_intraop.detach().item(),
        "intracorp": loss_intracorp.detach().item(),
    }
    return total_loss, loss_dict


# -------------------------
# Train
# -------------------------
def train():
    train_rows = load_jsonl(train_jsonl)
    num_intraop_classes, num_intracorp_classes = infer_track_vocab_sizes(train_rows)

    print("num_intraop_classes:", num_intraop_classes)
    print("num_intracorp_classes:", num_intracorp_classes)

    train_dataset = TaskBDataset(train_jsonl, max_frames_per_video=max_frames_per_video)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn
    )

    model = TaskBQwen25VLLoRA(
        model_name=model_name,
        num_phase_classes=num_phase_classes,
        num_instrument_classes=num_instrument_classes,
        num_operator_classes=num_operator_classes,
        num_intraop_classes=num_intraop_classes,
        num_intracorp_classes=num_intracorp_classes,
        num_slots=num_slots,
        lora_r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
    ).to(device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, eps=adam_eps)

    global_step = 0

    for epoch in range(num_epochs):
        model.train()

        running_total = 0.0
        steps = 0
        avg_phase = 0.0
        avg_obj = 0.0
        avg_instr = 0.0
        avg_op = 0.0
        avg_bbox = 0.0
        avg_intraop = 0.0
        avg_intracorp = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for sample in pbar:
            global_step += 1

            try:
                outputs = model(sample["frames"])
            except Exception as e:
                print("\n[FORWARD FAIL]")
                print("video:", sample["video_path"])
                print("frame_indices:", sample["frame_indices"])
                raise e

            loss, loss_dict = compute_taskb_loss(outputs, sample["anchor_targets"])

            if torch.isnan(loss) or torch.isinf(loss):
                print("\n[BAD LOSS]")
                print("video:", sample["video_path"])
                print("frame_indices:", sample["frame_indices"])
                raise ValueError("loss invalid")

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=grad_clip_norm)
            optimizer.step()

            running_total += loss.item()
            avg_phase += loss_dict["phase"]
            avg_obj += loss_dict["obj"]
            avg_instr += loss_dict["instr"]
            avg_op += loss_dict["op"]
            avg_bbox += loss_dict["bbox"]
            avg_intraop += loss_dict["intraop"]
            avg_intracorp += loss_dict["intracorp"]
            steps += 1

            pbar.set_postfix(
                total=f"{running_total/max(steps,1):.4f}",
                phase=f"{avg_phase/max(steps,1):.4f}",
                obj=f"{avg_obj/max(steps,1):.4f}",
                bbox=f"{avg_bbox/max(steps,1):.4f}",
                step=global_step
            )

        epoch_total = running_total / max(steps, 1)
        print(
            f"\nEpoch {epoch+1}/{num_epochs} done | "
            f"total={epoch_total:.4f} | "
            f"phase={avg_phase/max(steps,1):.4f} | "
            f"obj={avg_obj/max(steps,1):.4f} | "
            f"instr={avg_instr/max(steps,1):.4f} | "
            f"op={avg_op/max(steps,1):.4f} | "
            f"bbox={avg_bbox/max(steps,1):.4f} | "
            f"intraop={avg_intraop/max(steps,1):.4f} | "
            f"intracorp={avg_intracorp/max(steps,1):.4f}\n"
        )

    ckpt = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch_total_loss": epoch_total,
        "num_intraop_classes": num_intraop_classes,
        "num_intracorp_classes": num_intracorp_classes,
        "num_slots": num_slots,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
    }

    #torch.save(ckpt, os.path.join(save_dir, f"epoch_{epoch+1}.pt"))
    torch.save(ckpt, os.path.join(save_dir, "last.pt"))

    print("Training done.")
    print("Saved to:", save_dir)


if __name__ == "__main__":
    train()

num_intraop_classes: 10
num_intracorp_classes: 52
num usable rows: 17


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

trainable params: 921,600 || all params: 3,755,544,576 || trainable%: 0.0245


Epoch 1/20:   0%|          | 0/17 [00:00<?, ?it/s]

...


Epoch 18/20 done | total=10.3493 | phase=1.5195 | obj=0.5248 | instr=1.1622 | op=1.0313 | bbox=2.1642 | intraop=1.4780 | intracorp=2.4693



Epoch 19/20:   0%|          | 0/17 [00:00<?, ?it/s]


Epoch 19/20 done | total=10.2524 | phase=1.4999 | obj=0.5588 | instr=1.1540 | op=0.9106 | bbox=2.1397 | intraop=1.4869 | intracorp=2.5025



Epoch 20/20:   0%|          | 0/17 [00:00<?, ?it/s]


Epoch 20/20 done | total=10.0525 | phase=1.4826 | obj=0.5538 | instr=1.1251 | op=0.8804 | bbox=2.1177 | intraop=1.4577 | intracorp=2.4352

Training done.
Saved to: /content/drive/MyDrive/CholecTrack20/taskB_qwen25vl_lora


In [23]:
import os
import json
import time
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2_5_VLModel
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm


test_jsonl = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetune_long_video_10s.jsonl"
ckpt_path = "/content/drive/MyDrive/CholecTrack20/taskB_qwen25vl_lora/last.pt"
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
pred_out = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetuned_anchor_predictions.jsonl"

num_phase_classes = 7
num_instrument_classes = 7
num_operator_classes = 4
obj_threshold = 0.5
device = "cuda" if torch.cuda.is_available() else "cpu"

max_frames_per_video = 90
lora_r = 4
lora_alpha = 8
lora_dropout = 0.05


# -------------------------
# Utils
# -------------------------

def round_bbox(bbox, ndigits=4):
    return [round(float(x), ndigits) for x in bbox]


# -------------------------
# Dataset
# -------------------------
class TaskBTestDataset(Dataset):
    def __init__(self, jsonl_path, max_frames_per_video=None):
        self.rows = load_jsonl(jsonl_path)
        self.max_frames_per_video = max_frames_per_video

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        video_path = row["video"]
        frame_indices = row["frame_indices"]
        anchor_targets = row["anchor_targets"]

        if self.max_frames_per_video is not None:
            frame_indices = frame_indices[:self.max_frames_per_video]
            anchor_targets = anchor_targets[:self.max_frames_per_video]

        frames = read_frames_rgb(video_path, frame_indices)

        return {
            "video": video_path,
            "frames": frames,
            "frame_indices": frame_indices,
            "anchor_targets": anchor_targets,
            "clip_json_path": row.get("clip_json_path", None),
        }




# -------------------------
# Decode prediction
# -------------------------
def decode_taskb_outputs(outputs, frame_indices, obj_threshold=0.5):
    phase_logits = outputs["phase_logits"][0]
    obj_logits = outputs["obj_logits"][0]
    instr_logits = outputs["instr_logits"][0]
    op_logits = outputs["op_logits"][0]
    bbox_pred = outputs["bbox_pred"][0]
    intraop_logits = outputs["intraop_logits"][0]
    intracorp_logits = outputs["intracorp_logits"][0]

    phase_pred = phase_logits.argmax(dim=-1).detach().cpu().tolist()
    obj_prob = torch.sigmoid(obj_logits).detach().cpu()
    instr_pred = instr_logits.argmax(dim=-1).detach().cpu().tolist()
    op_pred = op_logits.argmax(dim=-1).detach().cpu().tolist()
    intraop_pred = intraop_logits.argmax(dim=-1).detach().cpu().tolist()
    intracorp_pred = intracorp_logits.argmax(dim=-1).detach().cpu().tolist()
    bbox_pred = bbox_pred.detach().cpu().tolist()

    frames = []

    for t, fid in enumerate(frame_indices):
        objects = []
        k = len(obj_prob[t])

        for s in range(k):
            if float(obj_prob[t][s].item()) >= obj_threshold:
                objects.append({
                    "instrument": int(instr_pred[t][s]),
                    "tool_bbox": round_bbox(bbox_pred[t][s]),
                    "operator": int(op_pred[t][s]),
                    "intraoperative_track": int(intraop_pred[t][s]),
                    "intracorporeal_track": int(intracorp_pred[t][s]),
                })

        frames.append({
            "frame_index": int(fid),
            "phase": int(phase_pred[t]),
            "objects": objects
        })

    return {"frames": frames}


@torch.no_grad()
def run_inference():
    dataset = TaskBTestDataset(test_jsonl, max_frames_per_video=max_frames_per_video)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

    ckpt = torch.load(ckpt_path, map_location=device)

    model = TaskBQwen25VLLoRA(
        model_name=model_name,
        num_phase_classes=num_phase_classes,
        num_instrument_classes=num_instrument_classes,
        num_operator_classes=num_operator_classes,
        num_intraop_classes=ckpt["num_intraop_classes"],
        num_intracorp_classes=ckpt["num_intracorp_classes"],
        num_slots=ckpt["num_slots"],
        lora_r=ckpt.get("lora_r", lora_r),
        lora_alpha=ckpt.get("lora_alpha", lora_alpha),
        lora_dropout=ckpt.get("lora_dropout", lora_dropout),
    ).to(device)

    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()

    rows = []

    for sample in tqdm(loader, desc="Task B inference"):
        outputs = model(sample["frames"])
        prediction = decode_taskb_outputs(
            outputs=outputs,
            frame_indices=sample["frame_indices"],
            obj_threshold=obj_threshold
        )

        ground_truth = {"frames": sample["anchor_targets"]}

        rows.append({
            "video": sample["video"],
            "ground_truth": ground_truth,
            "prediction": prediction,
            "clip_json_path": sample["clip_json_path"]
        })

    with open(pred_out, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("saved:", pred_out)


if __name__ == "__main__":
    run_inference()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Task B inference:   0%|          | 0/3 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetuned_anchor_predictions.jsonl


In [24]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)


pred_path = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/finetuned_anchor_predictions.jsonl"
output_dir = "/content/drive/MyDrive/CholecTrack20/CholecTrack20_15min_clips/testing/eval_finetuned_anchor_task"
iou_threshold = 0.5

os.makedirs(output_dir, exist_ok=True)


# -------------------------
# Utils
# -------------------------
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def bbox_xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]


def box_area_xyxy(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_iou_xywh(box1, box2):
    if box1 is None or box2 is None:
        return 0.0

    ax1, ay1, ax2, ay2 = bbox_xywh_to_xyxy(box1)
    bx1, by1, bx2, by2 = bbox_xywh_to_xyxy(box2)

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = box_area_xyxy([ax1, ay1, ax2, ay2])
    area_b = box_area_xyxy([bx1, by1, bx2, by2])
    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0
    return inter_area / union




# -------------------------
# Matching
# -------------------------


def hungarian_match_by_iou(gt_objects, pred_objects, iou_threshold=0.5):
    n_gt = len(gt_objects)
    n_pred = len(pred_objects)

    if n_gt == 0 and n_pred == 0:
        return [], [], []
    if n_gt == 0:
        return [], [], list(range(n_pred))
    if n_pred == 0:
        return [], list(range(n_gt)), []

    cost = np.zeros((n_gt, n_pred), dtype=np.float32)
    iou_mat = np.zeros((n_gt, n_pred), dtype=np.float32)

    for i, gt in enumerate(gt_objects):
        for j, pred in enumerate(pred_objects):
            iou = bbox_iou_xywh(gt["tool_bbox"], pred["tool_bbox"])
            iou_mat[i, j] = iou
            cost[i, j] = 1.0 - iou

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    matched_gt = set()
    matched_pred = set()

    for r, c in zip(row_ind.tolist(), col_ind.tolist()):
        iou = float(iou_mat[r, c])
        if iou >= iou_threshold:
            matches.append((r, c, iou))
            matched_gt.add(r)
            matched_pred.add(c)

    unmatched_gt = [i for i in range(n_gt) if i not in matched_gt]
    unmatched_pred = [j for j in range(n_pred) if j not in matched_pred]

    return matches, unmatched_gt, unmatched_pred


# -------------------------
# Main eval
# -------------------------
rows = load_jsonl(pred_path)

# B. Phase
phase_gt_all = []
phase_pred_all = []

# C. Detection
tp = 0
fp = 0
fn = 0
matched_ious = []
per_instr_ious = {k: [] for k in instrument_map.keys()}

# D. Matched-object classification
instr_gt_all = []
instr_pred_all = []

op_gt_all = []
op_pred_all = []

intraop_gt_all = []
intraop_pred_all = []

intracorp_gt_all = []
intracorp_pred_all = []

for row in rows:
    gt = row["ground_truth"]
    pred = row["prediction"]

    gt_frames = gt["frames"]
    pred_frames = pred["frames"]

    n = min(len(gt_frames), len(pred_frames))

    for i in range(n):
        gt_frame = gt_frames[i]
        pred_frame = pred_frames[i]

        # Phase
        phase_gt_all.append(int(gt_frame["phase"]))
        phase_pred_all.append(int(pred_frame["phase"]))

        # Detection / matching
        gt_objects = gt_frame["objects"]
        pred_objects = pred_frame["objects"]

        matches, unmatched_gt, unmatched_pred = hungarian_match_by_iou(
            gt_objects,
            pred_objects,
            iou_threshold=iou_threshold
        )

        tp += len(matches)
        fn += len(unmatched_gt)
        fp += len(unmatched_pred)

        for gt_idx, pred_idx, iou in matches:
            matched_ious.append(iou)

            gt_obj = gt_objects[gt_idx]
            pred_obj = pred_objects[pred_idx]

            gt_instr = int(gt_obj["instrument"])
            pred_instr = int(pred_obj["instrument"])

            if gt_instr in per_instr_ious:
                per_instr_ious[gt_instr].append(iou)

            instr_gt_all.append(gt_instr)
            instr_pred_all.append(pred_instr)

            op_gt_all.append(int(gt_obj["operator"]))
            op_pred_all.append(int(pred_obj["operator"]))

            intraop_gt_all.append(int(gt_obj["intraoperative_track"]))
            intraop_pred_all.append(int(pred_obj["intraoperative_track"]))

            intracorp_gt_all.append(int(gt_obj["intracorporeal_track"]))
            intracorp_pred_all.append(int(pred_obj["intracorporeal_track"]))


# -------------------------
# B. Phase metrics
# -------------------------
phase_accuracy = accuracy_score(phase_gt_all, phase_pred_all) if len(phase_gt_all) > 0 else 0.0
phase_macro_f1 = f1_score(phase_gt_all, phase_pred_all, average="macro", zero_division=0) if len(phase_gt_all) > 0 else 0.0

phase_labels = list(range(7))
phase_cm = confusion_matrix(phase_gt_all, phase_pred_all, labels=phase_labels) if len(phase_gt_all) > 0 else np.zeros((7, 7), dtype=int)

save_confusion_matrix(
    phase_cm,
    [phase_map[x] for x in phase_labels],
    "Task B Phase Confusion Matrix",
    os.path.join(output_dir, "phase_confusion_matrix.png")
)

# -------------------------
# C. Detection / localization metrics
# -------------------------
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_det = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
bbox_mean_iou = float(np.mean(matched_ious)) if len(matched_ious) > 0 else 0.0

per_instr_mean_iou = {}
for k, name in instrument_map.items():
    vals = per_instr_ious[k]
    per_instr_mean_iou[name] = float(np.mean(vals)) if len(vals) > 0 else 0.0

save_bar_chart(
    list(per_instr_mean_iou.keys()),
    list(per_instr_mean_iou.values()),
    "BBox IoU by Instrument",
    "Mean IoU",
    os.path.join(output_dir, "bbox_iou_by_instrument.png")
)

# -------------------------
# D. Matched-object classification metrics
# -------------------------
# Instrument
instrument_accuracy = accuracy_score(instr_gt_all, instr_pred_all) if len(instr_gt_all) > 0 else 0.0
instrument_report = classification_report(
    instr_gt_all,
    instr_pred_all,
    labels=list(instrument_map.keys()),
    target_names=[instrument_map[x] for x in instrument_map.keys()],
    zero_division=0,
    digits=4
) if len(instr_gt_all) > 0 else "No matched objects."

# Operator
operator_accuracy = accuracy_score(op_gt_all, op_pred_all) if len(op_gt_all) > 0 else 0.0
operator_report = classification_report(
    op_gt_all,
    op_pred_all,
    labels=list(operator_map.keys()),
    target_names=[operator_map[x] for x in operator_map.keys()],
    zero_division=0,
    digits=4
) if len(op_gt_all) > 0 else "No matched objects."

# Intraoperative track
intraop_exact_acc = accuracy_score(intraop_gt_all, intraop_pred_all) if len(intraop_gt_all) > 0 else 0.0
intraop_macro_f1 = f1_score(intraop_gt_all, intraop_pred_all, average="macro", zero_division=0) if len(intraop_gt_all) > 0 else 0.0
intraop_micro_f1 = f1_score(intraop_gt_all, intraop_pred_all, average="micro", zero_division=0) if len(intraop_gt_all) > 0 else 0.0

# Intracorporeal track
intracorp_exact_acc = accuracy_score(intracorp_gt_all, intracorp_pred_all) if len(intracorp_gt_all) > 0 else 0.0
intracorp_macro_f1 = f1_score(intracorp_gt_all, intracorp_pred_all, average="macro", zero_division=0) if len(intracorp_gt_all) > 0 else 0.0
intracorp_micro_f1 = f1_score(intracorp_gt_all, intracorp_pred_all, average="micro", zero_division=0) if len(intracorp_gt_all) > 0 else 0.0


# -------------------------
# Save summary
# -------------------------
summary = {
    "phase": {
        "accuracy": phase_accuracy,
        "macro_f1": phase_macro_f1,
    },
    "detection_localization": {
        "iou_threshold": iou_threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1_det,
        "bbox_mean_iou": bbox_mean_iou,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "per_instrument_mean_iou": per_instr_mean_iou,
    },
    "matched_object_classification": {
        "instrument": {
            "accuracy": instrument_accuracy,
        },
        "operator": {
            "accuracy": operator_accuracy,
        },
        "intraoperative_track": {
            "exact_match_accuracy": intraop_exact_acc,
            "macro_f1": intraop_macro_f1,
            "micro_f1": intraop_micro_f1,
        },
        "intracorporeal_track": {
            "exact_match_accuracy": intracorp_exact_acc,
            "macro_f1": intracorp_macro_f1,
            "micro_f1": intracorp_micro_f1,
        },
    },
}

with open(os.path.join(output_dir, "summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

with open(os.path.join(output_dir, "instrument_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(instrument_report)

with open(os.path.join(output_dir, "operator_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(operator_report)


# -------------------------
# Pretty print
# -------------------------
print("TASK B\n")

print("================================================================================")
print("B. PHASE")
print("================================================================================")
print(f"Accuracy: {phase_accuracy:.4f}")
print(f"Macro F1: {phase_macro_f1:.4f}")

print("\n================================================================================")
print("C. DETECTION / LOCALIZATION")
print("================================================================================")
print(f"Hungarian matching IoU threshold: {iou_threshold}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1_det:.4f}")
print(f"BBox mean IoU: {bbox_mean_iou:.4f}")

print("\nPer-instrument mean IoU:")
for name, val in per_instr_mean_iou.items():
    print(f"  {name}: {val:.4f}")

print("\n================================================================================")
print("D. MATCHED-OBJECT CLASSIFICATION")
print("================================================================================")

print("\nInstrument")
print(f"Accuracy: {instrument_accuracy:.4f}")
print(instrument_report)

print("\nOperator")
print(f"Accuracy: {operator_accuracy:.4f}")
print(operator_report)

print("\nIntraoperative Track")
print(f"Exact match accuracy: {intraop_exact_acc:.4f}")
print(f"Macro F1: {intraop_macro_f1:.4f}")
print(f"Micro F1: {intraop_micro_f1:.4f}")

print("\nIntracorporeal Track")
print(f"Exact match accuracy: {intracorp_exact_acc:.4f}")
print(f"Macro F1: {intracorp_macro_f1:.4f}")
print(f"Micro F1: {intracorp_micro_f1:.4f}")

print("\nSaved to:", output_dir)
print("- summary_metrics.json")
print("- instrument_classification_report.txt")
print("- operator_classification_report.txt")
print("- phase_confusion_matrix.png")
print("- bbox_iou_by_instrument.png")

TASK B

B. PHASE
Accuracy: 0.1481
Macro F1: 0.0877

C. DETECTION / LOCALIZATION
Hungarian matching IoU threshold: 0.5
Precision: 0.0020
Recall: 0.0018
F1: 0.0019
BBox mean IoU: 0.6973

Per-instrument mean IoU:
  grasper: 0.0000
  bipolar: 0.6973
  hook: 0.0000
  scissors: 0.0000
  clipper: 0.0000
  irrigator: 0.0000
  specimen-bag: 0.0000

D. MATCHED-OBJECT CLASSIFICATION

Instrument
Accuracy: 0.0000
              precision    recall  f1-score   support

     grasper     0.0000    0.0000    0.0000       0.0
     bipolar     0.0000    0.0000    0.0000       1.0
        hook     0.0000    0.0000    0.0000       0.0
    scissors     0.0000    0.0000    0.0000       0.0
     clipper     0.0000    0.0000    0.0000       0.0
   irrigator     0.0000    0.0000    0.0000       0.0
specimen-bag     0.0000    0.0000    0.0000       0.0

    accuracy                         0.0000       1.0
   macro avg     0.0000    0.0000    0.0000       1.0
weighted avg     0.0000    0.0000    0.0000       1.0
